# Side Effect Hybrid Evaluation

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 16.


In [ ]:
# ============================================================
# FINAL STAGE
# HYBRID UNIQUE-263 — 3-SEED FINAL EVALUATION + ENSEMBLE
#
# Seeds: 42, 43, 44
#
# IMPORTANT:
#   - NO TRAINING
#   - Threshold selection uses CALIBRATION ONLY
#   - Test is evaluated AFTER thresholds are frozen
#
# Threshold protocol:
#   Global thresholds: 0.05 -> 0.85 step 0.01
#   Per-label thresholds: 0.01 -> 0.95 step 0.01
#   MIN_CAL_SUPPORT = 3
#   Comparison operator = >=
#   Threshold tie -> smallest threshold
#   Method objective:
#       0.5 * Micro-F1 + 0.5 * Macro-F1
#   Method tie -> GLOBAL
#
# Final ensemble:
#   P = (P42 + P43 + P44) / 3
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json
import shutil
import warnings

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")


# ============================================================
# 2) CONFIG
# ============================================================

SEEDS = [42, 43, 44]

N_LABELS = 263

N_CAL = 28768
N_TEST = 28768

GLOBAL_GRID = np.round(
    np.arange(
        0.05,
        0.851,
        0.01
    ),
    2
)

PER_LABEL_GRID = np.round(
    np.arange(
        0.01,
        0.951,
        0.01
    ),
    2
)

MIN_CAL_SUPPORT = 3


# ============================================================
# 3) DATA PATHS
# ============================================================

TRAIN_DIR = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

STAGE2_DATA_DIR = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

Y_CAL_PATH = (
    STAGE2_DATA_DIR /
    "Y_calibration.npy"
)

Y_TEST_PATH = (
    STAGE2_DATA_DIR /
    "Y_test.npy"
)

LABEL_MAP_PATH = (
    TRAIN_DIR /
    "label_mapping.csv"
)


# ============================================================
# 4) OUTPUT
# ============================================================

OUT_DIR = Path(
    "/kaggle/working/"
    "FINAL_UNIQUE263_HYBRID_EVALUATION"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5) FILE CHECK
# ============================================================

print("=" * 120)
print("FINAL HYBRID UNIQUE-263 EVALUATION")
print("=" * 120)

for p in [
    Y_CAL_PATH,
    Y_TEST_PATH,
    LABEL_MAP_PATH
]:

    print(
        "" if p.exists() else "",
        p
    )

    if not p.exists():

        raise FileNotFoundError(
            p
        )


# ============================================================
# 6) LOAD TARGETS
# ============================================================

print("\nLoading targets...")

Y_cal = np.load(
    Y_CAL_PATH,
    mmap_mode="r"
)

Y_test = np.load(
    Y_TEST_PATH,
    mmap_mode="r"
)


label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


LABELS = (
    label_map[
        "label"
    ]
    .astype(str)
    .tolist()
)


print(
    "Y_cal:",
    Y_cal.shape
)

print(
    "Y_test:",
    Y_test.shape
)

print(
    "Labels:",
    len(LABELS)
)


assert Y_cal.shape == (
    N_CAL,
    N_LABELS
)

assert Y_test.shape == (
    N_TEST,
    N_LABELS
)

assert len(
    LABELS
) == N_LABELS


# ============================================================
# 7) FINAL UNIQUE-263 QA
# ============================================================

print("\n" + "=" * 120)
print("FINAL UNIQUE-263 QA")
print("=" * 120)

special_expected = {

    "anaphylaxis":
        (4, 4),

    "hyperglycemia":
        (4, 4),

    "tinnitus":
        (3, 3),
}


for label_name, expected in (
    special_expected.items()
):

    ids = (
        label_map.loc[
            label_map[
                "label"
            ].astype(str)
            ==
            label_name,
            "label_id"
        ]
        .astype(int)
        .tolist()
    )


    if len(ids) != 1:

        raise RuntimeError(
            f"Cannot uniquely find "
            f"{label_name}"
        )


    j = ids[0]


    observed = (

        int(
            np.asarray(
                Y_cal[:, j]
            ).sum()
        ),

        int(
            np.asarray(
                Y_test[:, j]
            ).sum()
        )
    )


    print(
        label_name,
        "| ID=",
        j,
        "| cal/test=",
        observed
    )


    if observed != expected:

        raise RuntimeError(
            f"{label_name}: "
            f"expected {expected}, "
            f"got {observed}"
        )


print(
    "\n Final UNIQUE-263 targets verified"
)


# ============================================================
# 8) FIND COMPLETE SEED DIRECTORY
# ============================================================

def find_seed_directory(seed):

    roots = [

        Path(
            "/kaggle/working"
        ),

        Path(
            "/kaggle/input"
        )
    ]


    candidates = []


    for root in roots:

        if not root.exists():

            continue


        for cal_file in root.rglob(
            "calibration_probabilities.npy"
        ):

            folder = (
                cal_file.parent
            )

            test_file = (
                folder /
                "test_probabilities.npy"
            )


            if not test_file.exists():

                continue


            folder_text = (
                str(folder)
                .lower()
            )


            if (
                f"seed_{seed}"
                not in
                folder_text
            ):

                continue


            # Ignore obvious partial directories
            if (
                "partial"
                in
                folder_text
            ):

                continue


            score = 0


            if (
                "/kaggle/working/"
                in
                str(folder)
            ):

                score += 100


            if (
                folder.name
                ==
                f"seed_{seed}"
            ):

                score += 50


            if (
                f"final_unique263_hybrid_seed_{seed}"
                in
                folder.name.lower()
            ):

                score += 40


            complete_file = (
                folder /
                "training_complete.json"
            )


            if complete_file.exists():

                try:

                    with open(
                        complete_file,
                        "r"
                    ) as f:

                        info = json.load(
                            f
                        )


                    if (
                        info.get(
                            "status"
                        )
                        ==
                        "complete"
                    ):

                        score += 30

                except:
                    pass


            candidates.append(
                (
                    score,
                    folder
                )
            )


    if not candidates:

        raise FileNotFoundError(
            f"\n Could not find a COMPLETE "
            f"Seed {seed} folder containing:\n"
            "calibration_probabilities.npy\n"
            "test_probabilities.npy"
        )


    candidates = sorted(

        candidates,

        key=lambda x:
            x[0],

        reverse=True
    )


    print(
        f"\nSeed {seed} candidates:"
    )


    for score, folder in candidates:

        print(
            f" score={score} ->",
            folder
        )


    chosen = (
        candidates[0][1]
    )


    print(
        " Selected:",
        chosen
    )


    return chosen


# ============================================================
# 9) LOCATE ALL 3 SEEDS
# ============================================================

print("\n" + "=" * 120)
print("LOCATING SEEDS")
print("=" * 120)


seed_dirs = {}


for seed in SEEDS:

    seed_dirs[
        seed
    ] = find_seed_directory(
        seed
    )


# ============================================================
# 10) LOAD PROBABILITY MATRICES
# ============================================================

seed_cal_probs = {}
seed_test_probs = {}


print("\n" + "=" * 120)
print("LOADING SEED PROBABILITIES")
print("=" * 120)


for seed in SEEDS:

    folder = (
        seed_dirs[
            seed
        ]
    )


    cal_path = (
        folder /
        "calibration_probabilities.npy"
    )


    test_path = (
        folder /
        "test_probabilities.npy"
    )


    P_cal = np.load(
        cal_path
    ).astype(
        np.float32
    )


    P_test = np.load(
        test_path
    ).astype(
        np.float32
    )


    print(
        f"\nSeed {seed}"
    )

    print(
        "Calibration:",
        P_cal.shape,
        "range=",
        (
            float(
                P_cal.min()
            ),
            float(
                P_cal.max()
            )
        )
    )

    print(
        "Test:",
        P_test.shape,
        "range=",
        (
            float(
                P_test.min()
            ),
            float(
                P_test.max()
            )
        )
    )


    assert P_cal.shape == (
        N_CAL,
        N_LABELS
    )

    assert P_test.shape == (
        N_TEST,
        N_LABELS
    )


    assert np.isfinite(
        P_cal
    ).all()


    assert np.isfinite(
        P_test
    ).all()


    assert (
        P_cal.min()
        >=
        0
    )


    assert (
        P_cal.max()
        <=
        1
    )


    assert (
        P_test.min()
        >=
        0
    )


    assert (
        P_test.max()
        <=
        1
    )


    seed_cal_probs[
        seed
    ] = P_cal


    seed_test_probs[
        seed
    ] = P_test


print(
    "\n All 3 seed probability matrices loaded"
)


# ============================================================
# 11) FAST METRIC HELPER
# ============================================================

def micro_macro_from_predictions(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=np.uint8
    )


    y_pred = np.asarray(
        y_pred,
        dtype=np.uint8
    )


    tp = np.sum(

        (
            y_true
            ==
            1
        )

        &

        (
            y_pred
            ==
            1
        ),

        axis=0
    )


    fp = np.sum(

        (
            y_true
            ==
            0
        )

        &

        (
            y_pred
            ==
            1
        ),

        axis=0
    )


    fn = np.sum(

        (
            y_true
            ==
            1
        )

        &

        (
            y_pred
            ==
            0
        ),

        axis=0
    )


    denom_label = (
        2 * tp
        +
        fp
        +
        fn
    )


    label_f1 = np.divide(

        2 * tp,

        denom_label,

        out=np.zeros(
            N_LABELS,
            dtype=np.float64
        ),

        where=(
            denom_label
            >
            0
        )
    )


    macro_f1 = float(
        np.mean(
            label_f1
        )
    )


    TP = int(
        tp.sum()
    )

    FP = int(
        fp.sum()
    )

    FN = int(
        fn.sum()
    )


    denom_micro = (
        2 * TP
        +
        FP
        +
        FN
    )


    micro_f1 = (

        2 * TP
        /
        denom_micro

        if denom_micro
        >
        0

        else 0.0
    )


    return (

        float(
            micro_f1
        ),

        float(
            macro_f1
        )
    )


# ============================================================
# 12) FULL METRICS
# ============================================================

def full_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=np.uint8
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.uint8
    )


    return {

        "exact_match":
            float(
                np.mean(
                    np.all(
                        y_true
                        ==
                        y_pred,
                        axis=1
                    )
                )
            ),

        "micro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),

        "micro_precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "micro_recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
            ),

        "macro_precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "macro_recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "hamming_loss":
            float(
                hamming_loss(
                    y_true,
                    y_pred
                )
            )
    }


# ============================================================
# 13) GLOBAL THRESHOLD SEARCH
# ============================================================

def select_global_threshold(
    y_cal,
    probs
):

    rows = []


    best_threshold = None
    best_objective = -1.0
    best_micro = None
    best_macro = None


    for threshold in GLOBAL_GRID:

        pred = (

            probs

            >=

            threshold

        ).astype(
            np.uint8
        )


        micro_f1, macro_f1 = (
            micro_macro_from_predictions(
                y_cal,
                pred
            )
        )


        objective = (
            0.5
            *
            micro_f1

            +

            0.5
            *
            macro_f1
        )


        rows.append({

            "threshold":
                float(
                    threshold
                ),

            "micro_f1":
                micro_f1,

            "macro_f1":
                macro_f1,

            "objective":
                float(
                    objective
                )
        })


        # strict > means first/smallest
        # threshold wins ties
        if (
            objective
            >
            best_objective
            +
            1e-15
        ):

            best_objective = (
                objective
            )

            best_threshold = float(
                threshold
            )

            best_micro = (
                micro_f1
            )

            best_macro = (
                macro_f1
            )


    return {

        "threshold":
            best_threshold,

        "objective":
            float(
                best_objective
            ),

        "micro_f1":
            float(
                best_micro
            ),

        "macro_f1":
            float(
                best_macro
            ),

        "table":
            pd.DataFrame(
                rows
            )
    }


# ============================================================
# 14) PER-LABEL BEST BINARY F1
# ============================================================

def best_label_threshold(
    y_true,
    probs
):

    y_true = np.asarray(
        y_true,
        dtype=np.uint8
    )


    probs = np.asarray(
        probs,
        dtype=np.float32
    )


    predictions = (

        probs[
            :,
            None
        ]

        >=

        PER_LABEL_GRID[
            None,
            :
        ]

    )


    truth = (

        y_true[
            :,
            None
        ]

        ==
        1

    )


    tp = np.sum(
        predictions
        &
        truth,
        axis=0
    )


    fp = np.sum(
        predictions
        &
        (~truth),
        axis=0
    )


    fn = (
        int(
            y_true.sum()
        )

        -

        tp
    )


    denominator = (

        2 * tp

        +

        fp

        +

        fn

    )


    f1s = np.divide(

        2 * tp,

        denominator,

        out=np.zeros(
            len(
                PER_LABEL_GRID
            ),
            dtype=np.float64
        ),

        where=(
            denominator
            >
            0
        )
    )


    # np.argmax -> first max
    # grids ascending -> smallest threshold
    best_idx = int(
        np.argmax(
            f1s
        )
    )


    return (

        float(
            PER_LABEL_GRID[
                best_idx
            ]
        ),

        float(
            f1s[
                best_idx
            ]
        )
    )


# ============================================================
# 15) PER-LABEL THRESHOLD SELECTION
# ============================================================

def select_per_label_thresholds(
    y_cal,
    probs,
    fallback_global
):

    thresholds = np.full(
        N_LABELS,
        fallback_global,
        dtype=np.float32
    )


    individual_cal_f1 = np.zeros(
        N_LABELS,
        dtype=np.float32
    )


    supports = np.asarray(
        y_cal.sum(
            axis=0
        )
    ).reshape(-1)


    for j in range(
        N_LABELS
    ):

        support = int(
            supports[
                j
            ]
        )


        if support >= MIN_CAL_SUPPORT:

            threshold, binary_f1 = (
                best_label_threshold(

                    np.asarray(
                        y_cal[
                            :,
                            j
                        ]
                    ),

                    probs[
                        :,
                        j
                    ]
                )
            )


            thresholds[
                j
            ] = threshold


            individual_cal_f1[
                j
            ] = binary_f1


        else:

            pred = (

                probs[
                    :,
                    j
                ]

                >=

                fallback_global

            ).astype(
                np.uint8
            )


            individual_cal_f1[
                j
            ] = f1_score(

                np.asarray(
                    y_cal[
                        :,
                        j
                    ]
                ),

                pred,

                zero_division=0
            )


    pred_cal = (

        probs

        >=

        thresholds[
            None,
            :
        ]

    ).astype(
        np.uint8
    )


    micro_f1, macro_f1 = (
        micro_macro_from_predictions(

            y_cal,

            pred_cal

        )
    )


    objective = (

        0.5
        *
        micro_f1

        +

        0.5
        *
        macro_f1

    )


    return {

        "thresholds":
            thresholds,

        "individual_cal_f1":
            individual_cal_f1,

        "micro_f1":
            float(
                micro_f1
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "objective":
            float(
                objective
            )
    }


# ============================================================
# 16) CALIBRATE ONE SYSTEM
# ============================================================

def calibrate_system(
    system_name,
    cal_probs
):

    print("\n" + "=" * 120)

    print(
        "CALIBRATING:",
        system_name
    )

    print("=" * 120)


    # --------------------------------------------------------
    # GLOBAL
    # --------------------------------------------------------

    global_result = (
        select_global_threshold(

            Y_cal,

            cal_probs

        )
    )


    print(
        "\nBest global threshold:",
        global_result[
            "threshold"
        ]
    )


    print(
        "Global calibration objective:",
        round(
            global_result[
                "objective"
            ],
            6
        )
    )


    print(
        "Global Micro-F1:",
        round(
            global_result[
                "micro_f1"
            ],
            6
        )
    )


    print(
        "Global Macro-F1:",
        round(
            global_result[
                "macro_f1"
            ],
            6
        )
    )


    # --------------------------------------------------------
    # PER LABEL
    # --------------------------------------------------------

    per_label_result = (
        select_per_label_thresholds(

            Y_cal,

            cal_probs,

            global_result[
                "threshold"
            ]

        )
    )


    print(
        "\nPer-label calibration objective:",
        round(
            per_label_result[
                "objective"
            ],
            6
        )
    )


    print(
        "Per-label Micro-F1:",
        round(
            per_label_result[
                "micro_f1"
            ],
            6
        )
    )


    print(
        "Per-label Macro-F1:",
        round(
            per_label_result[
                "macro_f1"
            ],
            6
        )
    )


    # --------------------------------------------------------
    # METHOD SELECTION
    #
    # Tie -> GLOBAL
    # --------------------------------------------------------

    if (

        per_label_result[
            "objective"
        ]

        >

        global_result[
            "objective"
        ]

        +

        1e-15

    ):

        selected_method = (
            "per_label"
        )


        selected_thresholds = (
            per_label_result[
                "thresholds"
            ]
        )


        selected_objective = (
            per_label_result[
                "objective"
            ]
        )


    else:

        selected_method = (
            "global"
        )


        selected_thresholds = np.full(

            N_LABELS,

            global_result[
                "threshold"
            ],

            dtype=np.float32
        )


        selected_objective = (
            global_result[
                "objective"
            ]
        )


    print(
        "\n SELECTED METHOD:",
        selected_method.upper()
    )


    print(
        "Selected calibration objective:",
        round(
            selected_objective,
            6
        )
    )


    # save global search
    safe_name = (
        system_name
        .replace(
            " ",
            "_"
        )
        .lower()
    )


    global_result[
        "table"
    ].to_csv(

        OUT_DIR
        /
        f"{safe_name}_global_threshold_search.csv",

        index=False
    )


    threshold_df = pd.DataFrame({

        "label_id":
            np.arange(
                N_LABELS
            ),

        "label":
            LABELS,

        "calibration_support":
            np.asarray(
                Y_cal.sum(
                    axis=0
                )
            ).reshape(-1),

        "threshold":
            selected_thresholds,

        "per_label_cal_f1":
            per_label_result[
                "individual_cal_f1"
            ]
    })


    threshold_df.to_csv(

        OUT_DIR
        /
        f"{safe_name}_selected_thresholds.csv",

        index=False
    )


    return {

        "system":
            system_name,

        "method":
            selected_method,

        "global_threshold":
            global_result[
                "threshold"
            ],

        "global_objective":
            global_result[
                "objective"
            ],

        "per_label_objective":
            per_label_result[
                "objective"
            ],

        "selected_objective":
            selected_objective,

        "thresholds":
            selected_thresholds
    }


# ============================================================
# 17) EVALUATE TEST AFTER FREEZE
# ============================================================

def evaluate_test(
    system_name,
    test_probs,
    calibration_info
):

    thresholds = (
        calibration_info[
            "thresholds"
        ]
    )


    test_pred = (

        test_probs

        >=

        thresholds[
            None,
            :
        ]

    ).astype(
        np.uint8
    )


    metrics = full_metrics(
        Y_test,
        test_pred
    )


    metrics[
        "system"
    ] = system_name


    metrics[
        "threshold_method"
    ] = (
        calibration_info[
            "method"
        ]
    )


    metrics[
        "global_threshold"
    ] = (
        calibration_info[
            "global_threshold"
        ]
    )


    metrics[
        "calibration_objective"
    ] = (
        calibration_info[
            "selected_objective"
        ]
    )


    # --------------------------------------------------------
    # Per-label test metrics
    # --------------------------------------------------------

    precision, recall, f1, support = (
        precision_recall_fscore_support(

            Y_test,

            test_pred,

            average=None,

            zero_division=0
        )
    )


    predicted_support = (
        test_pred.sum(
            axis=0
        )
    )


    per_label_df = pd.DataFrame({

        "label_id":
            np.arange(
                N_LABELS
            ),

        "label":
            LABELS,

        "threshold":
            thresholds,

        "test_support":
            support.astype(
                int
            ),

        "predicted_positive":
            predicted_support.astype(
                int
            ),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1
    })


    safe_name = (
        system_name
        .replace(
            " ",
            "_"
        )
        .lower()
    )


    per_label_df.to_csv(

        OUT_DIR
        /
        f"{safe_name}_test_per_label.csv",

        index=False
    )


    return (
        metrics,
        per_label_df
    )


# ============================================================
# 18) INDIVIDUAL SEED EVALUATION
# ============================================================

individual_rows = []

individual_per_label = {}


for seed in SEEDS:

    system_name = (
        f"Seed {seed}"
    )


    calibration_info = (
        calibrate_system(

            system_name,

            seed_cal_probs[
                seed
            ]
        )
    )


    metrics, per_label = (
        evaluate_test(

            system_name,

            seed_test_probs[
                seed
            ],

            calibration_info
        )
    )


    individual_rows.append(
        metrics
    )


    individual_per_label[
        seed
    ] = per_label


# ============================================================
# 19) INDIVIDUAL RESULTS
# ============================================================

individual_df = pd.DataFrame(
    individual_rows
)


metric_order = [

    "system",

    "threshold_method",

    "global_threshold",

    "calibration_objective",

    "exact_match",

    "micro_f1",

    "macro_f1",

    "weighted_f1",

    "micro_precision",

    "micro_recall",

    "macro_precision",

    "macro_recall",

    "hamming_loss"
]


individual_df = individual_df[
    metric_order
]


individual_df.to_csv(

    OUT_DIR /
    "individual_seed_test_metrics.csv",

    index=False
)


print("\n" + "=" * 120)
print("INDIVIDUAL SEED TEST RESULTS")
print("=" * 120)


display_df = (
    individual_df.copy()
)


for col in [

    "exact_match",

    "micro_f1",

    "macro_f1",

    "weighted_f1",

    "micro_precision",

    "micro_recall",

    "macro_precision",

    "macro_recall"

]:

    display_df[
        col
    ] = (

        display_df[
            col
        ]

        *
        100
    )


print(
    display_df.to_string(
        index=False
    )
)


# ============================================================
# 20) SEED MEAN ± SD
# ============================================================

numeric_metrics = [

    "exact_match",

    "micro_f1",

    "macro_f1",

    "weighted_f1",

    "micro_precision",

    "micro_recall",

    "macro_precision",

    "macro_recall",

    "hamming_loss"
]


summary_rows = []


for metric in numeric_metrics:

    values = (
        individual_df[
            metric
        ]
        .astype(float)
        .values
    )


    summary_rows.append({

        "metric":
            metric,

        "mean":
            float(
                values.mean()
            ),

        "sample_sd":
            float(
                values.std(
                    ddof=1
                )
            )
    })


seed_summary_df = pd.DataFrame(
    summary_rows
)


seed_summary_df.to_csv(

    OUT_DIR /
    "individual_seed_mean_sd.csv",

    index=False
)


print("\n" + "=" * 120)
print("3-SEED MEAN ± SAMPLE SD")
print("=" * 120)


for _, row in (
    seed_summary_df.iterrows()
):

    metric = row[
        "metric"
    ]


    mean = row[
        "mean"
    ]


    sd = row[
        "sample_sd"
    ]


    if metric != "hamming_loss":

        print(
            f"{metric:20s}: "
            f"{mean*100:.4f}% "
            f"± {sd*100:.4f}%"
        )

    else:

        print(
            f"{metric:20s}: "
            f"{mean:.8f} "
            f"± {sd:.8f}"
        )


# ============================================================
# 21) EQUAL PROBABILITY ENSEMBLE
# ============================================================

print("\n" + "=" * 120)
print("BUILDING EQUAL 3-SEED ENSEMBLE")
print("=" * 120)


ensemble_cal = (

    seed_cal_probs[
        42
    ]

    +

    seed_cal_probs[
        43
    ]

    +

    seed_cal_probs[
        44
    ]

) / 3.0


ensemble_test = (

    seed_test_probs[
        42
    ]

    +

    seed_test_probs[
        43
    ]

    +

    seed_test_probs[
        44
    ]

) / 3.0


ensemble_cal = (
    ensemble_cal.astype(
        np.float32
    )
)


ensemble_test = (
    ensemble_test.astype(
        np.float32
    )
)


np.save(

    OUT_DIR /
    "ensemble_calibration_probabilities.npy",

    ensemble_cal
)


np.save(

    OUT_DIR /
    "ensemble_test_probabilities.npy",

    ensemble_test
)


# ============================================================
# 22) CALIBRATE ENSEMBLE
# ============================================================

ensemble_calibration = (
    calibrate_system(

        "Equal Ensemble",

        ensemble_cal
    )
)


# ============================================================
# 23) FINAL ENSEMBLE TEST
# ============================================================

ensemble_metrics, ensemble_per_label = (
    evaluate_test(

        "Equal Ensemble",

        ensemble_test,

        ensemble_calibration
    )
)


ensemble_df = pd.DataFrame(
    [
        ensemble_metrics
    ]
)


ensemble_df.to_csv(

    OUT_DIR /
    "FINAL_ensemble_test_metrics.csv",

    index=False
)


# ============================================================
# 24) PRINT FINAL ENSEMBLE
# ============================================================

print("\n" + "=" * 120)
print("FINAL HYBRID EQUAL ENSEMBLE — TEST RESULTS")
print("=" * 120)


print(
    "Threshold method :",
    ensemble_metrics[
        "threshold_method"
    ]
)


print(
    "Global threshold :",
    ensemble_metrics[
        "global_threshold"
    ]
)


print(
    "Exact Match      :",
    f"{ensemble_metrics['exact_match']*100:.4f}%"
)


print(
    "Micro-F1         :",
    f"{ensemble_metrics['micro_f1']*100:.4f}%"
)


print(
    "Macro-F1         :",
    f"{ensemble_metrics['macro_f1']*100:.4f}%"
)


print(
    "Weighted-F1      :",
    f"{ensemble_metrics['weighted_f1']*100:.4f}%"
)


print(
    "Micro Precision  :",
    f"{ensemble_metrics['micro_precision']*100:.4f}%"
)


print(
    "Micro Recall     :",
    f"{ensemble_metrics['micro_recall']*100:.4f}%"
)


print(
    "Macro Precision  :",
    f"{ensemble_metrics['macro_precision']*100:.4f}%"
)


print(
    "Macro Recall     :",
    f"{ensemble_metrics['macro_recall']*100:.4f}%"
)


print(
    "Hamming Loss     :",
    f"{ensemble_metrics['hamming_loss']:.8f}"
)


# ============================================================
# 25) SPECIAL NEW LABEL RESULTS
# ============================================================

print("\n" + "=" * 120)
print("NEW REPLACEMENT LABELS — ENSEMBLE TEST")
print("=" * 120)


special_labels = [

    "anaphylaxis",

    "hyperglycemia",

    "tinnitus"
]


special_result = (
    ensemble_per_label[
        ensemble_per_label[
            "label"
        ].isin(
            special_labels
        )
    ]
    .copy()
)


print(
    special_result[
        [
            "label_id",

            "label",

            "threshold",

            "test_support",

            "predicted_positive",

            "precision",

            "recall",

            "f1"
        ]
    ].to_string(
        index=False
    )
)


special_result.to_csv(

    OUT_DIR /
    "replacement_labels_ensemble_test.csv",

    index=False
)


# ============================================================
# 26) COMPARE AGAINST PREVIOUS CURRENT BEST
# ============================================================

PREVIOUS_BEST = {

    "exact_match":
        0.764217,

    "micro_f1":
        0.916328,

    "macro_f1":
        0.920570,

    "weighted_f1":
        0.916363,

    "micro_precision":
        0.910681,

    "micro_recall":
        0.922045,

    "macro_precision":
        0.915423,

    "macro_recall":
        0.929944,

    "hamming_loss":
        0.00394674
}


comparison_rows = []


for metric, old_value in (
    PREVIOUS_BEST.items()
):

    new_value = float(
        ensemble_metrics[
            metric
        ]
    )


    if metric == "hamming_loss":

        # negative delta = improvement
        delta = (
            new_value
            -
            old_value
        )


        comparison_rows.append({

            "metric":
                metric,

            "previous_best":
                old_value,

            "new_hybrid":
                new_value,

            "delta":
                delta,

            "direction":
                (
                    "IMPROVED"
                    if new_value
                    <
                    old_value
                    else
                    (
                        "SAME"
                        if np.isclose(
                            new_value,
                            old_value
                        )
                        else
                        "WORSE"
                    )
                )
        })


    else:

        delta = (

            new_value

            -

            old_value

        )


        comparison_rows.append({

            "metric":
                metric,

            "previous_best":
                old_value,

            "new_hybrid":
                new_value,

            "delta":
                delta,

            "direction":
                (
                    "IMPROVED"
                    if new_value
                    >
                    old_value
                    else
                    (
                        "SAME"
                        if np.isclose(
                            new_value,
                            old_value
                        )
                        else
                        "WORSE"
                    )
                )
        })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df.to_csv(

    OUT_DIR /
    "comparison_vs_previous_best.csv",

    index=False
)


print("\n" + "=" * 120)
print("NEW HYBRID vs PREVIOUS BEST ENSEMBLE")
print("=" * 120)


for _, row in (
    comparison_df.iterrows()
):

    metric = row[
        "metric"
    ]


    old = row[
        "previous_best"
    ]


    new = row[
        "new_hybrid"
    ]


    delta = row[
        "delta"
    ]


    direction = row[
        "direction"
    ]


    if metric != "hamming_loss":

        print(
            f"{metric:20s} "
            f"| OLD={old*100:.4f}% "
            f"| NEW={new*100:.4f}% "
            f"| Δ={delta*100:+.4f} pp "
            f"| {direction}"
        )


    else:

        print(
            f"{metric:20s} "
            f"| OLD={old:.8f} "
            f"| NEW={new:.8f} "
            f"| Δ={delta:+.8f} "
            f"| {direction}"
        )


# ============================================================
# 27) SAVE RUN SUMMARY JSON
# ============================================================

summary_json = {

    "seeds":
        SEEDS,

    "threshold_protocol": {

        "global_grid":
            "0.05 to 0.85 step 0.01",

        "per_label_grid":
            "0.01 to 0.95 step 0.01",

        "min_calibration_support":
            MIN_CAL_SUPPORT,

        "comparison_operator":
            ">=",

        "threshold_tie_rule":
            "smallest threshold",

        "method_objective":
            "0.5 * Micro-F1 + 0.5 * Macro-F1",

        "method_tie_rule":
            "global"
    },

    "ensemble":
        "equal probability average",

    "test_used_for_threshold_selection":
        False,

    "final_ensemble_metrics":
        {
            key:
                (
                    float(value)
                    if isinstance(
                        value,
                        (
                            float,
                            np.floating
                        )
                    )
                    else
                    value
                )

            for key, value
            in ensemble_metrics.items()
        }
}


with open(

    OUT_DIR /
    "FINAL_EVALUATION_SUMMARY.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(
        summary_json,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 28) ZIP ALL FINAL EVALUATION OUTPUTS
# ============================================================

ZIP_PATH = shutil.make_archive(

    "/kaggle/working/"
    "FINAL_UNIQUE263_HYBRID_EVALUATION",

    "zip",

    root_dir=
        OUT_DIR
)


print("\n" + "=" * 120)
print("FINAL EVALUATION COMPLETE")
print("=" * 120)


print(
    "Results folder:"
)

print(
    OUT_DIR
)


print(
    "\nZIP:"
)

print(
    ZIP_PATH
)


print(
    "\n NO MORE TRAINING."
)

print(
    " Final 3-seed hybrid evaluation complete."
)